# 1 SETTINGS

In [ ]:
pip install pulp

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.7/17.7 MB 54.2 MB/s eta 0:00:00


In [ ]:
# IMPORT CÁC THƯ VIỆN CẦN THIẾT, VÀ CẤP QUYỀN CHO GOOGLE COLAB
import pandas as pd
import numpy as np
import pulp
import datetime
import math

import gspread
import gspread_dataframe as gd

from google.colab import data_table, auth
data_table.enable_dataframe_formatter()
auth.authenticate_user()

from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# VỊ TRÍ CÁC DATASET
gs_link = 'https://docs.google.com/spreadsheets/d/1_0KR5UIilGliYdqqoMYGJjYKpRQYTs0RAOhZEpQxTUs'
booking_sheet = 'Lô cần xếp'
booking_range = 'A2:L'

capacity_sheet = 'Lô cần xếp'
capacity_range = 'R2:T'

result_sheet = 'Suggest Solution'
result_date_range = 'A3:O'
result_remain_range = 'S3:U'

In [ ]:
# CÀI ĐẶT NHỮNG NGÀY NGHỈ LỄ SẼ BỊ BỎ RA KHÔNG XẾP LỊCH
#list_day_off = ['2023-12-31','2023-12-30']

list_day_off = []   # SỬA DÒNG NÀY 'YYYY-MM-DD',...
list_day_off = pd.to_datetime(list_day_off, format='%Y-%m-%d')

In [ ]:
today = pd.to_datetime(datetime.datetime.today().strftime('%Y-%m-%d'))

# 2 READ DATA

## 2.1 df raw

In [ ]:
# ĐỌC VÀ XỬ LÝ DATA CÁC LÔ HÀNG
df_booking = pd.DataFrame.from_records(gc.open_by_url(gs_link).worksheet(booking_sheet).get(booking_range))
df_booking.columns = df_booking.iloc[0,:]
df_booking = df_booking.iloc[1:,:]
df_booking.reset_index(drop=True, inplace=True)
df_booking = df_booking.drop_duplicates()
df_booking = df_booking.applymap(lambda x: x.strip() if isinstance(x, str) else x)

df_booking = df_booking[~df_booking['Material Type'].isnull()]
df_booking = df_booking[df_booking['Material Type']!='']
df_booking[['Pallet Needed', 'Loader Needed']] = df_booking[['Pallet Needed', 'Loader Needed']].apply(lambda x: x.str.replace(',', '').astype(float))
df_booking[['Earliest Date', 'Target 1', 'Target 2']] = df_booking[['Earliest Date', 'Target 1', 'Target 2']].apply(lambda col: pd.to_datetime(col, format='%d-%b-%Y', errors='coerce'))
df_booking['Bill Cont/Seal'] = df_booking['Bill'] + ' ' + df_booking['Cont/Seal']
df_booking = df_booking.reset_index().rename(columns={'index': 'raw index'})    # CỘT RAW INDEX ĐỂ REFER MERGE SAU NÀY DỄ HƠN

df_booking

<ipython-input-16-f1a1b5e2f123>:7: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_booking = df_booking.applymap(lambda x: x.strip() if isinstance(x, str) else x)


,raw index,Material Type,Supplier,Invoice,PO,Bill,Target 1,Target 2,Earliest Date,Cont/Seal,Pallet Needed,Loader Needed,Priority,Bill Cont/Seal
0,0,Cước,PEDEX,5440029214,4503139944,15530330111,NaT,NaT,NaT,,1.0,0.01,None,15530330111
1,1,Rpet,YANGZHOU SANXING,INYZTS2409001,,COAU7253218470,2024-10-05,2024-10-07,NaT,CBHU4388468 / NA615582,25.0,0.35,None,COAU7253218470 CBHU4388468 / NA615582
2,2,Rpet,YANGZHOU SANXING,INYZTS2409001,,COAU7253218470,2024-10-05,2024-10-07,NaT,CSLU1092386 / NE632204,27.0,0.38,None,COAU7253218470 CSLU1092386 / NE632204
3,3,Rpet,YANGZHOU SANXING,INYZTS2409001,,COAU7253218470,2024-10-05,2024-10-07,NaT,CSNU1564406 / NE633434,27.0,0.38,None,COAU7253218470 CSNU1564406 / NE633434
4,4,Rpet,YANGZHOU SANXING,INYZTS2409001,,COAU7253218470,2024-10-05,2024-10-07,NaT,CSNU1749360 / NE632295,27.0,0.38,x,COAU7253218470 CSNU1749360 / NE632295
5,5,Rpet,YANGZHOU SANXING,INYZTS2409001,,COAU7253218470,2024-10-05,2024-10-07,NaT,OOLU1439700 / NA613798,25.0,0.35,x,COAU7253218470 OOLU1439700 / NA613798
6,6,Rpet,YANGZHOU SANXING,INYZTS2409001,,COAU7253218470,2024-10-05,2024-10-07,NaT,TEMU3252552 / NE633436,28.0,0.39,None,COAU7253218470 TEMU3252552 / NE633436
7,7,Decal,SENTIEN,ST-I2409028,,TPEB53778,NaT,NaT,2024-10-01,LCL,14.0,0.19,None,TPEB53778 LCL
8,8,Nhựa,NANTONG,PLM6034,4503179350,RJNTSE24090238,NaT,NaT,2024-10-02,LCL,5.0,0.07,None,RJNTSE24090238 LCL
9,9,Nhựa,BASELL,9925363170,4503174037,EGLV295400051346,NaT,NaT,NaT,TRHU6754922 / MA486949,36.0,0.50,None,EGLV295400051346 TRHU6754922 / MA486949


In [ ]:
df_booking.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28 entries, 0 to 27
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   raw index       28 non-null     int64         
 1   Material Type   28 non-null     object        
 2   Supplier        28 non-null     object        
 3   Invoice         28 non-null     object        
 4   PO              28 non-null     object        
 5   Bill            28 non-null     object        
 6   Target 1        9 non-null      datetime64[ns]
 7   Target 2        9 non-null      datetime64[ns]
 8   Earliest Date   10 non-null     datetime64[ns]
 9   Cont/Seal       28 non-null     object        
 10  Pallet Needed   28 non-null     float64       
 11  Loader Needed   28 non-null     float64       
 12  Priority        2 non-null      object        
 13  Bill Cont/Seal  28 non-null     object        
dtypes: datetime64[ns](3), float64(2), int64(1), object(8)
memory

In [ ]:
# LƯU LẠI DATA LÔ HÀNG Ở TRẠNG THÁI BAN ĐẦU
df_booking_raw = df_booking.copy()

## 2.2 df special

In [ ]:
# Booking special là AIR, LCL hoặc GẤP
#df_booking_special = df_booking[(df_booking['Cont/Seal'].isin(['LCL', 'AIR'])) | (df_booking['Priority'] == 'x')]
df_booking_special = df_booking[
    (df_booking['Cont/Seal'].isin(['LCL', 'AIR'])) |
    (df_booking['Priority'].str.lower().str.contains('gấp') | df_booking['Priority'].str.lower().str.contains('gap'))
]
df_booking_special = df_booking_special[~df_booking_special['Earliest Date'].isnull()]

# Nếu ES là hôm nay thì chọn ngày mai, nều ES là tương lai thì chọn ES
df_booking_special['Chosen Date'] = df_booking_special['Earliest Date'].apply(
    lambda x: x + datetime.timedelta(days=1) if x == today else x
)
df_booking_special

,raw index,Material Type,Supplier,Invoice,PO,Bill,Target 1,Target 2,Earliest Date,Cont/Seal,Pallet Needed,Loader Needed,Priority,Bill Cont/Seal,Chosen Date
7,7,Decal,SENTIEN,ST-I2409028,,TPEB53778,NaT,NaT,2024-10-01,LCL,14.0,0.19,None,TPEB53778 LCL,2024-10-01
8,8,Nhựa,NANTONG,PLM6034,4503179350,RJNTSE24090238,NaT,NaT,2024-10-02,LCL,5.0,0.07,None,RJNTSE24090238 LCL,2024-10-03
11,11,Cước,WUXI,XD240910-001,4503178386,EURFL24903005SGN,NaT,NaT,2024-10-01,LCL,1.0,0.01,None,EURFL24903005SGN LCL,2024-10-01
13,13,Cước,SHANGHAI COMPASS,SHJD24-361B,4503176478,SHCLI24906662J,NaT,NaT,2024-10-01,LCL,1.0,0.01,None,SHCLI24906662J LCL,2024-10-01
14,14,Cước,SHANGHAI COMPASS,SHJD24-374,4503178465,SHCLI24906662A,NaT,NaT,2024-10-02,LCL,8.0,0.11,None,SHCLI24906662A LCL,2024-10-03
22,22,Dây nhôm,HAINING,JH24-067H,4503176570,CL2409067VIA,NaT,NaT,2024-10-01,LCL,2.0,0.03,None,CL2409067VIA LCL,2024-10-01
24,24,Cước,K.R.,IEP2409032,4503178462,BKKC08353,NaT,NaT,2024-10-01,LCL,5.0,0.07,None,BKKC08353 LCL,2024-10-01


## 2.3 df booking

In [ ]:
# Các booking xếp bình thường
df_booking = df_booking[~((df_booking['Cont/Seal'].isin(['LCL', 'AIR'])) | (df_booking['Priority'] == 'x'))]
df_booking.head()

,raw index,Material Type,Supplier,Invoice,PO,Bill,Target 1,Target 2,Earliest Date,Cont/Seal,Pallet Needed,Loader Needed,Priority,Bill Cont/Seal
0,0,Cước,PEDEX,5440029214,4503139944,15530330111,NaT,NaT,NaT,,1.0,0.01,None,15530330111
1,1,Rpet,YANGZHOU SANXING,INYZTS2409001,,COAU7253218470,2024-10-05,2024-10-07,NaT,CBHU4388468 / NA615582,25.0,0.35,None,COAU7253218470 CBHU4388468 / NA615582
2,2,Rpet,YANGZHOU SANXING,INYZTS2409001,,COAU7253218470,2024-10-05,2024-10-07,NaT,CSLU1092386 / NE632204,27.0,0.38,None,COAU7253218470 CSLU1092386 / NE632204
3,3,Rpet,YANGZHOU SANXING,INYZTS2409001,,COAU7253218470,2024-10-05,2024-10-07,NaT,CSNU1564406 / NE633434,27.0,0.38,None,COAU7253218470 CSNU1564406 / NE633434
6,6,Rpet,YANGZHOU SANXING,INYZTS2409001,,COAU7253218470,2024-10-05,2024-10-07,NaT,TEMU3252552 / NE633436,28.0,0.39,None,COAU7253218470 TEMU3252552 / NE633436


In [ ]:
df_booking = df_booking.dropna(subset=['Earliest Date', 'Target 1', 'Target 2'])
df_booking.head()

,raw index,Material Type,Supplier,Invoice,PO,Bill,Target 1,Target 2,Earliest Date,Cont/Seal,Pallet Needed,Loader Needed,Priority,Bill Cont/Seal
15,15,Nhựa,TOP POLYMER,TPE240914001,,SITGSHSGP075396,2024-10-09,2024-10-09,2024-09-26,BMOU4376770 / SITC439324,29.0,0.40,None,SITGSHSGP075396 BMOU4376770 / SITC439324
16,16,Nhựa,TOP POLYMER,TPE240914001,,SITGSHSGP075396,2024-10-09,2024-10-09,2024-09-26,TLLU3515515 / SITE584025,15.0,0.21,None,SITGSHSGP075396 TLLU3515515 / SITE584025
21,21,Nhựa,SK,4000064033,4503179339,KMTCPUSI216938,2024-10-11,2024-10-11,2024-10-01,KMTU7398764 / KSB991925,23.0,0.32,None,KMTCPUSI216938 KMTU7398764 / KSB991925


## 2.4 df capacity

In [ ]:
# ĐỌC VÀ XỬ LÝ DATA CAPACITY MỖI NGÀY
df_capacity = pd.DataFrame.from_records(gc.open_by_url(gs_link).worksheet(capacity_sheet).get(capacity_range))
df_capacity.columns = df_capacity.iloc[0,:]
df_capacity = df_capacity.iloc[1:,:]
df_capacity.reset_index(drop=True, inplace=True)
df_capacity = df_capacity.drop_duplicates()
df_capacity = df_capacity.applymap(lambda x: x.strip() if isinstance(x, str) else x)

df_capacity = df_capacity[~df_capacity['Loader Remain'].isnull()]
df_capacity = df_capacity[df_capacity['Loader Remain']!='']
df_capacity[['Pallet Remain', 'Loader Remain']] = df_capacity[['Pallet Remain', 'Loader Remain']].apply(lambda x: x.str.replace(',', '').astype(float))
df_capacity['Date'] = df_capacity['Date'].apply(lambda x: pd.to_datetime(x, format='%d/%m/%Y'))

df_capacity.head()

<ipython-input-25-f340ff468bf6>:7: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_capacity = df_capacity.applymap(lambda x: x.strip() if isinstance(x, str) else x)


,Date,Pallet Remain,Loader Remain
0,2024-09-24,140.0,0.0
1,2024-09-25,80.0,2.0
2,2024-09-26,53.0,0.0
3,2024-09-27,19.0,0.0
4,2024-09-28,740.0,0.0


In [ ]:
df_capacity.info()

<class 'pandas.core.frame.DataFrame'>
Index: 25 entries, 0 to 24
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Date           25 non-null     datetime64[ns]
 1   Pallet Remain  25 non-null     float64       
 2   Loader Remain  25 non-null     float64       
dtypes: datetime64[ns](1), float64(2)
memory usage: 800.0 bytes


In [ ]:
# Exclude past day
df_capacity = df_capacity[df_capacity['Date'] > today]

In [ ]:
df_booking_special.head()

,raw index,Material Type,Supplier,Invoice,PO,Bill,Target 1,Target 2,Earliest Date,Cont/Seal,Pallet Needed,Loader Needed,Priority,Bill Cont/Seal,Chosen Date
7,7,Decal,SENTIEN,ST-I2409028,,TPEB53778,NaT,NaT,2024-10-01,LCL,14.0,0.19,None,TPEB53778 LCL,2024-10-01
8,8,Nhựa,NANTONG,PLM6034,4503179350,RJNTSE24090238,NaT,NaT,2024-10-02,LCL,5.0,0.07,None,RJNTSE24090238 LCL,2024-10-03
11,11,Cước,WUXI,XD240910-001,4503178386,EURFL24903005SGN,NaT,NaT,2024-10-01,LCL,1.0,0.01,None,EURFL24903005SGN LCL,2024-10-01
13,13,Cước,SHANGHAI COMPASS,SHJD24-361B,4503176478,SHCLI24906662J,NaT,NaT,2024-10-01,LCL,1.0,0.01,None,SHCLI24906662J LCL,2024-10-01
14,14,Cước,SHANGHAI COMPASS,SHJD24-374,4503178465,SHCLI24906662A,NaT,NaT,2024-10-02,LCL,8.0,0.11,None,SHCLI24906662A LCL,2024-10-03


In [ ]:
df_booking_special_sum = df_booking_special.groupby('Chosen Date', as_index=False).agg({'Pallet Needed': 'sum', 'Loader Needed': 'sum'})
df_booking_special_sum

,Chosen Date,Pallet Needed,Loader Needed
0,2024-10-01,23.0,0.31
1,2024-10-03,13.0,0.18


In [ ]:
df_capacity = df_capacity.merge(df_booking_special_sum, how='left', left_on='Date', right_on='Chosen Date')
df_capacity[['Pallet Needed', 'Loader Needed']] = df_capacity[['Pallet Needed', 'Loader Needed']].fillna(0.0)
df_capacity.head()

,Date,Pallet Remain,Loader Remain,Chosen Date,Pallet Needed,Loader Needed
0,2024-10-03,676.0,5.0,2024-10-03,13.0,0.18
1,2024-10-04,72.0,2.2,NaT,0.0,0.00
2,2024-10-05,79.0,4.0,NaT,0.0,0.00
3,2024-10-06,539.0,5.0,NaT,0.0,0.00
4,2024-10-07,-517.0,0.0,NaT,0.0,0.00


In [ ]:
# Trừ capacity của Booking đặc biệt, gấp ra
df_capacity['Pallet Remain'] = df_capacity['Pallet Remain'] - df_capacity['Pallet Needed']
df_capacity['Loader Remain'] = df_capacity['Loader Remain'] - df_capacity['Loader Needed']
df_capacity = df_capacity.drop(columns=['Chosen Date', 'Pallet Needed', 'Loader Needed'])
df_capacity

,Date,Pallet Remain,Loader Remain
0,2024-10-03,663.0,4.82
1,2024-10-04,72.0,2.20
2,2024-10-05,79.0,4.00
3,2024-10-06,539.0,5.00
4,2024-10-07,-517.0,0.00
5,2024-10-08,-42.0,1.50
6,2024-10-09,406.0,1.00
7,2024-10-10,532.0,2.00
8,2024-10-11,504.0,2.00
9,2024-10-12,756.0,1.00


# 3 CHOSE DATE

In [ ]:
# ĐỔI TÊN DATAFRAME LẠI CHO DỄ HIỂU KHI XÂY DỰNG MODEL
po_df = df_booking
date_df = df_capacity

In [ ]:
# Filter out dates with negative Pallet or Loader Remain or Sunday or Holidays or the past
valid_dates = date_df[(date_df['Pallet Remain'] >= 0) &
                      (date_df['Loader Remain'] >= 0)]
valid_dates = valid_dates[(~valid_dates['Date'].isin(list_day_off)) &
                          (valid_dates['Date'].dt.dayofweek != 6) &
                          (valid_dates['Date'] > today)]

In [ ]:
'''
Model chính
'''

'\nModel chính\n'

In [ ]:
# Create a list of all valid decisions (all combinations of POs and valid Dates)
decisions = [(po, date) for po in po_df['Bill Cont/Seal'] for date in valid_dates['Date']]

# Create binary decision variables for each valid combination
x = pulp.LpVariable.dicts('x', decisions, cat='Binary')

# Create the model
model = pulp.LpProblem("PO_Optimization", pulp.LpMinimize)

# Build cost function based on the criteria
cost = {}
for _, row in po_df.iterrows():
    for date in valid_dates['Date']:
        if date <= row['Target 1']:
            cost[(row['Bill Cont/Seal'], date)] = 0 # miễn phí khi trước hạn EDO
        elif date <= row['Target 2']:
            days_diff_1 = (date - row['Target 1']).days
            cost[(row['Bill Cont/Seal'], date)] = 10 * days_diff_1 # tốn tiền ít khi từ EDO đến DEM
        else:
            days_diff_1 = (row['Target 2'] - row['Target 1']).days
            days_diff_2 = (date - row['Target 2']).days
            cost[(row['Bill Cont/Seal'], date)] = 10 * days_diff_1 + 1000 * days_diff_2 # tốn nhiều tiền khi lố DEM

# Objective: Minimize total cost
model += pulp.lpSum(x[po, date] * cost[po, date] for po, date in decisions), "Total_Cost"

# Constraint: Each PO should be assigned to one and only one date
for po in po_df['Bill Cont/Seal']:
    model += pulp.lpSum(x[po, date] for date in valid_dates['Date']) == 1, f"One_date_{po}"

# Constraint: Ensure assigned date is on or after Earliest Date
for po, row in po_df.iterrows():
    for date in valid_dates['Date']:
        if date < row['Earliest Date']:
            model += x[(row['Bill Cont/Seal'], date)] == 0

# Constraint: Resource Pallet/Loader constraints for each date
for date in valid_dates['Date']:
    model += pulp.lpSum(x[po, date] * po_df.loc[po_df['Bill Cont/Seal'] == po, 'Pallet Needed'].values[0] for po in po_df['Bill Cont/Seal']) <= valid_dates.loc[valid_dates['Date'] == date, 'Pallet Remain'].values[0], f"Pallet_constraint_{date}"
    model += pulp.lpSum(x[po, date] * po_df.loc[po_df['Bill Cont/Seal'] == po, 'Loader Needed'].values[0] for po in po_df['Bill Cont/Seal']) <= valid_dates.loc[valid_dates['Date'] == date, 'Loader Remain'].values[0], f"Loader_constraint_{date}"

# Solve the problem
model.solve()

1

In [ ]:
# Retrieve the chosen dates and costs
chosen_dates = {}
cost_details = []
total_cost = 0
for po, date in decisions:
    if x[po, date].value() == 1:
        chosen_dates[po] = date
        po_cost = cost[(po, date)]
        total_cost += po_cost
        cost_details.append({'PO': po, 'Chosen Date': date, 'Cost': po_cost})

# Add the chosen dates to the PO DataFrame
po_df['Chosen Date'] = po_df['Bill Cont/Seal'].map(chosen_dates)

# Calculate remaining capacity
remaining_pallet = valid_dates.set_index('Date')['Pallet Remain'].to_dict()
remaining_loader = valid_dates.set_index('Date')['Loader Remain'].to_dict()

for po, date in chosen_dates.items():
    remaining_pallet[date] -= po_df.loc[po_df['Bill Cont/Seal'] == po, 'Pallet Needed'].values[0]
    remaining_loader[date] -= po_df.loc[po_df['Bill Cont/Seal'] == po, 'Loader Needed'].values[0]

In [ ]:
cost_details[1]

{'PO': 'SITGSHSGP075396 TLLU3515515 / SITE584025',
 'Chosen Date': Timestamp('2024-10-03 00:00:00'),
 'Cost': 0}

In [ ]:
df_cost = pd.DataFrame(cost_details, columns=['PO', 'Chosen Date', 'Cost'])
df_cost = df_cost.rename(columns={'PO': 'Bill Cont/Seal'})
df_cost.head()

,Bill Cont/Seal,Chosen Date,Cost
0,SITGSHSGP075396 BMOU4376770 / SITC439324,2024-10-03,0
1,SITGSHSGP075396 TLLU3515515 / SITE584025,2024-10-03,0
2,KMTCPUSI216938 KMTU7398764 / KSB991925,2024-10-05,0


In [ ]:
# remaining_pallet
# remaining_loader

In [ ]:
# Convert dictionaries to DataFrames
df_pallet = pd.DataFrame(list(remaining_pallet.items()), columns=['Date', 'Pallet Remain'])
df_loader = pd.DataFrame(list(remaining_loader.items()), columns=['Date', 'Loader Remain'])

# Merge DataFrames on the 'Date' column
df_remain = pd.merge(df_pallet, df_loader, on='Date')
df_remain.head()

,Date,Pallet Remain,Loader Remain
0,2024-10-03,619.0,4.21
1,2024-10-04,72.0,2.20
2,2024-10-05,56.0,3.68
3,2024-10-09,406.0,1.00
4,2024-10-10,532.0,2.00


In [ ]:
# LẤY LẠI NHỮNG NGÀY ĐÃ LOẠI RA KHỎI DANH SÁCH CHỌN, GỘP LẠI VÀ TRẢ VỀ KẾT QUẢ ĐÀY ĐỦ
df_remain = pd.concat([df_remain, df_capacity[~df_capacity['Date'].isin(df_remain['Date'])]], ignore_index=True)
df_remain = df_remain.sort_values(by='Date').reset_index(drop=True)
df_remain

,Date,Pallet Remain,Loader Remain
0,2024-10-03,619.0,4.21
1,2024-10-04,72.0,2.20
2,2024-10-05,56.0,3.68
3,2024-10-06,539.0,5.00
4,2024-10-07,-517.0,0.00
5,2024-10-08,-42.0,1.50
6,2024-10-09,406.0,1.00
7,2024-10-10,532.0,2.00
8,2024-10-11,504.0,2.00
9,2024-10-12,756.0,1.00


In [ ]:
# LỌC RA CÁC NGÀY KẾT QUẢ CỦA CẢ AIR, LCL, GẤP VÀ CÁC BOOKING XẾP BÌNH THƯỜNG
df_chosen = pd.concat([po_df[['raw index', 'Chosen Date']], df_booking_special[['raw index', 'Chosen Date']]], ignore_index=True)
df_chosen.head(1)

,raw index,Chosen Date
0,15,2024-10-03


In [ ]:
# NỐI CỘT DATE VÀ COST VÀO BẢNG KẾT QUẢ
df_booking_raw = df_booking_raw.merge(df_chosen, how='left', on='raw index')
df_booking_raw = df_booking_raw.drop(columns=['raw index'])
df_booking_raw = df_booking_raw.merge(df_cost, how='left', on=['Bill Cont/Seal','Chosen Date'])
df_booking_raw

,Material Type,Supplier,Invoice,PO,Bill,Target 1,Target 2,Earliest Date,Cont/Seal,Pallet Needed,Loader Needed,Priority,Bill Cont/Seal,Chosen Date,Cost
0,Cước,PEDEX,5440029214,4503139944,15530330111,NaT,NaT,NaT,,1.0,0.01,None,15530330111,NaT,NaN
1,Rpet,YANGZHOU SANXING,INYZTS2409001,,COAU7253218470,2024-10-05,2024-10-07,NaT,CBHU4388468 / NA615582,25.0,0.35,None,COAU7253218470 CBHU4388468 / NA615582,NaT,NaN
2,Rpet,YANGZHOU SANXING,INYZTS2409001,,COAU7253218470,2024-10-05,2024-10-07,NaT,CSLU1092386 / NE632204,27.0,0.38,None,COAU7253218470 CSLU1092386 / NE632204,NaT,NaN
3,Rpet,YANGZHOU SANXING,INYZTS2409001,,COAU7253218470,2024-10-05,2024-10-07,NaT,CSNU1564406 / NE633434,27.0,0.38,None,COAU7253218470 CSNU1564406 / NE633434,NaT,NaN
4,Rpet,YANGZHOU SANXING,INYZTS2409001,,COAU7253218470,2024-10-05,2024-10-07,NaT,CSNU1749360 / NE632295,27.0,0.38,x,COAU7253218470 CSNU1749360 / NE632295,NaT,NaN
5,Rpet,YANGZHOU SANXING,INYZTS2409001,,COAU7253218470,2024-10-05,2024-10-07,NaT,OOLU1439700 / NA613798,25.0,0.35,x,COAU7253218470 OOLU1439700 / NA613798,NaT,NaN
6,Rpet,YANGZHOU SANXING,INYZTS2409001,,COAU7253218470,2024-10-05,2024-10-07,NaT,TEMU3252552 / NE633436,28.0,0.39,None,COAU7253218470 TEMU3252552 / NE633436,NaT,NaN
7,Decal,SENTIEN,ST-I2409028,,TPEB53778,NaT,NaT,2024-10-01,LCL,14.0,0.19,None,TPEB53778 LCL,2024-10-01,NaN
8,Nhựa,NANTONG,PLM6034,4503179350,RJNTSE24090238,NaT,NaT,2024-10-02,LCL,5.0,0.07,None,RJNTSE24090238 LCL,2024-10-03,NaN
9,Nhựa,BASELL,9925363170,4503174037,EGLV295400051346,NaT,NaT,NaT,TRHU6754922 / MA486949,36.0,0.50,None,EGLV295400051346 TRHU6754922 / MA486949,NaT,NaN


# 4 PRINT RESULT

In [ ]:
# IN BẢNG KẾT QUẢ CHỌN NGÀY
gc.open_by_url(gs_link).values_clear(f"{result_sheet}!{result_date_range}")

worksheet = gc.open_by_url(gs_link ).worksheet(result_sheet)
cell_range = worksheet.range(result_date_range)
values = df_booking_raw.astype(str).replace({'NaT': '', 'None': '', 'nan': ''}).values.flatten()

# Assign the values to the cells in the range
for i, cell in enumerate(cell_range):
    if i < len(values):  # Make sure we don't go past the DataFrame's number of values
        cell.value = values[i]
    else:
        cell.value = ""  # Assign empty string to the remaining cells if any

# Update the cells in the worksheet
worksheet.update_cells(cell_range)

{'spreadsheetId': '1_0KR5UIilGliYdqqoMYGJjYKpRQYTs0RAOhZEpQxTUs',
 'updatedRange': "'Suggest Solution'!A3:O1000",
 'updatedRows': 998,
 'updatedColumns': 15,
 'updatedCells': 14970}

In [ ]:
# IN BẢNG REMAIN CAPACITY SAU KHI CHỌN NGÀY NHƯ BẢNG KẾT QUẢ
gc.open_by_url(gs_link).values_clear(f"{result_sheet}!{result_remain_range}")

worksheet = gc.open_by_url(gs_link ).worksheet(result_sheet)
cell_range = worksheet.range(result_remain_range)
values = df_remain.astype(str).values.flatten()

# Assign the values to the cells in the range
for i, cell in enumerate(cell_range):
    if i < len(values):  # Make sure we don't go past the DataFrame's number of values
        cell.value = values[i]
    else:
        cell.value = ""  # Assign empty string to the remaining cells if any

# Update the cells in the worksheet
worksheet.update_cells(cell_range)

{'spreadsheetId': '1_0KR5UIilGliYdqqoMYGJjYKpRQYTs0RAOhZEpQxTUs',
 'updatedRange': "'Suggest Solution'!S3:U1000",
 'updatedRows': 998,
 'updatedColumns': 3,
 'updatedCells': 2994}

In [ ]:
print('Done')

Done
